# Stage 2 — Multi-Channel Forecasting & Model Comparison

The first stage predicted sales from **Instagram ad spend alone**. Here we:

1. Add more channels: **TV ads**, **email spend** and **seasonality**.​
2. Compare **Multiple Linear Regression, Ridge, Polynomial Regression and Random Forest**.
3. Score every model with **5-fold cross-validation** (R² and MAE) so the ranking is robust.
4. Export the **best model** with `joblib` and serve it from a small **Streamlit app**.

## 1. Setup & load data

`season` is a categorical column, so it becomes one-hot dummies (`season_Q2`, `season_Q3`, `season_Q4`; Q1 is the reference quarter).

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

df = pd.read_csv("data/sales_data_multi.csv")
X = pd.get_dummies(df, columns=["season"], drop_first=True).drop(columns=["month", "sales"])
y = df["sales"]
feature_names = list(X.columns)

print("Input features:", feature_names)
print("Feature matrix shape:", X.shape)
X.head()

Input features: ['instagram_ad_spend', 'tv_ad_spend', 'email_spend', 'season_Q2', 'season_Q3', 'season_Q4']
Feature matrix shape: (96, 6)


,instagram_ad_spend,tv_ad_spend,email_spend,season_Q2,season_Q3,season_Q4
0,1810.89,11409.19,2721.17,False,False,False
1,3827.50,9695.74,1972.68,False,False,False
2,3061.98,2457.54,1149.28,False,False,False
3,2595.30,3942.05,1177.79,True,False,False
4,1046.07,2565.73,2232.68,True,False,False


## 2. Models to compare

- **Multiple Linear Regression** — interpretable: one coefficient per channel.
- **Ridge Regression** — linear with L2 regularization for stability.
- **Polynomial Regression (degree 2)** — captures non-linear channel interactions.
- **Random Forest** — flexible tree ensemble, no linearity assumption.

In [2]:
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10),
    "Polynomial Regression (deg 2)": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("lr", LinearRegression()),
    ]),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
}

rows = []
trained = {}
for name, model in models.items():
    cv = cross_validate(
        model, X, y, cv=5,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error"},
    )
    rows.append({
        "Model": name,
        "CV R²": round(cv["test_r2"].mean(), 3),
        "CV R² std": round(cv["test_r2"].std(), 3),
        "CV MAE ($)": round(-cv["test_mae"].mean()),
    })
    model.fit(X, y)
    trained[name] = model

results = pd.DataFrame(rows).sort_values("CV MAE ($)")
results

,Model,CV R²,CV R² std,CV MAE ($)
0,Multiple Linear Regression,0.947,0.012,19311
1,Ridge Regression,0.944,0.012,20498
2,Polynomial Regression (deg 2),0.925,0.019,21938
3,Random Forest,0.919,0.021,24599


## 3. What the numbers say

- **Multiple Linear Regression wins** with the lowest cross-validated MAE (~$19.3k/month) and the best R² (~0.95).
- Ridge is close, polynomial and Random Forest overfit this small simulated dataset.
- Because it is simple, linear regression also stays **interpretable** — which matters for budget decisions.

## 4. Interpret the winning model

Each coefficient answers **"how much more monthly sales does $1 of this channel buy?"**

In [3]:
best_name = results.iloc[0]["Model"]
best = trained[best_name]

coefs = pd.Series(best.coef_, index=feature_names).sort_values(key=np.abs, ascending=False)
print(f"Best model: {best_name}")
print(f"\nCoefficients (sales impact per unit spend):")
print(coefs.round(4).to_string())
print(f"\nIntercept: {best.intercept_:,.2f}")

Best model: Multiple Linear Regression

Coefficients (sales impact per unit spend):
season_Q4             38102.9929
season_Q3             10896.6124
season_Q2              6111.9928
instagram_ad_spend       93.9837
tv_ad_spend               3.1278
email_spend               2.4187

Intercept: -3,219.20


**Insights:**

- Instagram ads return more per dollar (~$94) than TV (~$3.13) or email (~$2.42) in this simulated data.
- Q4 (holiday season) adds ~$38k of monthly sales on top of the budget effects — that is the seasonality signal.

## 5. Export for the prediction app

The bundle stores the fitted model together with the exact feature column order, so the Streamlit app can rebuild the input row correctly.

In [4]:
joblib.dump({"model": best, "features": feature_names}, "output/sales_model.joblib")
print("Saved bundle to output/sales_model.joblib")

Saved bundle to output/sales_model.joblib


In [5]:
example = pd.DataFrame([{
    "instagram_ad_spend": 2500, "tv_ad_spend": 12000, "email_spend": 2000,
    "season_Q2": 0, "season_Q3": 0, "season_Q4": 1,
}])[feature_names]
print(f"Q4 forecast for $2.5k IG + $12k TV + $2k email: ${best.predict(example)[0]:,.0f}/month")

Q4 forecast for $2.5k IG + $12k TV + $2k email: $312,214/month


## 6. Conclusion

1. Adding channels and seasonality improved the forecast and told us **where marketing money does the most work**.
2. Cross-validation (not a single train/test split) made the model ranking trustworthy.
3. The winning model is exported and served by a 40-line **Streamlit app** (`app.py`):

   ```bash
   streamlit run app.py
   ```